In [24]:
import numpy as np
from joblib import load
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import regularizers
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.decomposition import PCA
import joblib
import numpy as np
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import load_model
from collections import Counter
from xgboost import XGBClassifier
from sklearn import tree
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import RobustScaler, LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier
)
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd
from sklearn.model_selection import train_test_split
pd.set_option('display.max_columns',None)
warnings.filterwarnings('ignore')
%matplotlib inline

In [2]:
data_train = pd.read_csv("KDDTrain+.txt", header=None)
data_test = pd.read_csv("KDDTest+.txt", header=None)

columns = (['duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent','hot'
,'num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root','num_file_creations'
,'num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count','srv_count','serror_rate'
,'srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count'
,'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate'
,'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate','outcome','level'])
data_train.columns = columns

In [3]:
columns = (['duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent','hot'
,'num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root','num_file_creations'
,'num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count','srv_count','serror_rate'
,'srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count'
,'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate'
,'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate','outcome','level'])

In [4]:
data_train.columns = columns
data_test.columns = columns

In [5]:
attack_mapping = {
    'normal': 'normal',
    'neptune': 'DoS',
    'smurf': 'DoS',
    'back': 'DoS',
    'teardrop': 'DoS',
    'pod': 'DoS',
    'land': 'DoS',
    'apache2': 'DoS',
    'mailbomb': 'DoS',
    'processtable': 'DoS',
    'udpstorm': 'DoS',
    'nuked': 'DoS',
    'worm': 'DoS',

    'ipsweep': 'Probe',
    'portsweep': 'Probe',
    'nmap': 'Probe',
    'satan': 'Probe',
    'mscan': 'Probe',
    'saint': 'Probe',
    'xsnoop': 'Probe',
    'snmpgetattack': 'Probe',
    'snmpguess': 'Probe',
    'httptunnel': 'Probe',

    'warezclient': 'R2L',
    'guess_passwd': 'R2L',
    'ftp_write': 'R2L',
    'multihop': 'R2L',
    'imap': 'R2L',
    'warezmaster': 'R2L',
    'phf': 'R2L',
    'spy': 'R2L',
    'sendmail': 'R2L',
    'secrect': 'R2L',

    'rootkit': 'U2R',
    'buffer_overflow': 'U2R',
    'loadmodule': 'U2R',
    'perl': 'U2R',
    'ps': 'U2R',
    'sqlattack': 'U2R',
    'xterm': 'U2R',
    'named': 'U2R',
    'xlock': 'U2R'
}

unmapped_attacks = set(data_train['outcome'].unique()) - set(attack_mapping.keys())
if unmapped_attacks:
    print(f"Warning: The following attack types in 'outcome' are not in the provided mapping: {unmapped_attacks}. They will be mapped to 'Other_Attack'.")
    for attack in unmapped_attacks:
        attack_mapping[attack] = 'Other_Attack'

data_train['attack_class'] = data_train['outcome'].map(attack_mapping)
data_test['attack_class'] = data_test['outcome'].map(attack_mapping)

print("\nValue counts for 'attack_class' after mapping:")
print(data_train['attack_class'].value_counts())
print("\nValue counts for 'attack_class' in test set after mapping:")
print(data_test['attack_class'].value_counts())


Value counts for 'attack_class' after mapping:
attack_class
normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64

Value counts for 'attack_class' in test set after mapping:
attack_class
normal    9711
DoS       7460
Probe     3067
R2L       2213
U2R         93
Name: count, dtype: int64


In [6]:
data_train = pd.get_dummies(
    data_train,
    columns=['protocol_type', 'service', 'flag'],
    drop_first=True
)
data_test = pd.get_dummies(
    data_test,
    columns=['protocol_type', 'service', 'flag'],
    drop_first=True
)

In [7]:
data_train, data_test = data_train.align(
    data_test,
    join='left',
    axis=1,
    fill_value=0
)

In [8]:
data_train_final=data_train.copy()
data_test_final=data_test.copy()

In [9]:
le = LabelEncoder()

data_train_final['attack_class'] = le.fit_transform(data_train_final['attack_class'])
data_test_final['attack_class'] = le.transform(data_test_final['attack_class'])

In [28]:
X_train = data_train_final.drop(['attack_class', 'outcome'], axis=1)
y_train = data_train_final['attack_class']
X_test = data_test_final.drop(['attack_class', 'outcome'], axis=1)
y_test = data_test_final['attack_class']

In [11]:
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'DoS': np.int64(0), 'Probe': np.int64(1), 'R2L': np.int64(2), 'U2R': np.int64(3), 'normal': np.int64(4)}


In [12]:
xgb_model = load('xgb_model_standalone.pkl')
catboost_model = load('catboost_model_standalone.pkl')
adaboost_model = load('adaboost_model_standalone.pkl')
randomforest_model = load('random_forest_smote_model.joblib')

svm_model=load('svm_model.pkl')
nb_model=load('naive_bayes_model.pkl')
knn_model=load('knn_model.pkl')
logistic_model=load('logistic_regression_model.pkl')
stand_scaler=load('standard_scaler.pkl')

mlp_model=load_model('mlp_ids_model.keras')
cnn_model = load_model('cnn_ids_model.keras')

scaler =load(
    "scaler.pkl"
)

le =load(
    "label_encoder.pkl"
)

encoder = load_model(
    "encoder_model.keras"
)

autoencoder = load_model(
    "autoencoder_model.keras"
)

xgb_auto_model = XGBClassifier()

xgb_auto_model.load_model(
    "xgb_model.json"
)
xgb_model_robust = XGBClassifier()

xgb_model_robust.load_model(
    "robust_xgb_model.json"
)

print("ALL MODELS LOADED SUCCESSFULLY!")


ALL MODELS LOADED SUCCESSFULLY!


In [13]:
X_test_scaled = scaler.transform(X_test)

In [14]:
X_test_scaled = X_test_scaled.astype(np.float32)
X_test_encoded = encoder.predict(
    X_test_scaled
)
X_test_encoded = X_test_encoded.astype(np.float32)

X_test_fused = np.concatenate(

    [
        X_test_scaled,
        X_test_encoded
    ],

    axis=1
)

print(X_test_fused.shape)

705/705 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step
(22544, 168)


In [25]:
y_pred = xgb_auto_model.predict(
    X_test_fused
)
hybrid_results = {}
hybrid_results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average='weighted'),
    "Recall": recall_score(y_test, y_pred, average='weighted'),
    "F1 Score": f1_score(y_test, y_pred, average='weighted'),
    "Macro F1": f1_score(y_test, y_pred, average='macro'),
    "Classification Report": classification_report(y_test, y_pred),
    "Confusion Matrix": confusion_matrix(y_test, y_pred)
}

print("\n====================================")
print("Hybrid MODEL RESULTS")
print("====================================")

print(f"Accuracy : {hybrid_results['Accuracy']:.4f}")
print(f"Precision: {hybrid_results['Precision']:.4f}")
print(f"Recall   : {hybrid_results['Recall']:.4f}")
print(f"F1 Score : {hybrid_results['F1 Score']:.4f}")
print(f"Macro F1 : {hybrid_results['Macro F1']:.4f}")

print("\nClassification Report:\n")
print(hybrid_results["Classification Report"])

print("\nConfusion Matrix:\n")
print(hybrid_results["Confusion Matrix"])


Hybrid MODEL RESULTS
Accuracy : 0.8252
Precision: 0.8507
Recall   : 0.8252
F1 Score : 0.8142
Macro F1 : 0.6838

Classification Report:

              precision    recall  f1-score   support

           0       0.96      0.84      0.90      7460
           1       0.86      0.70      0.77      3067
           2       0.95      0.33      0.49      2213
           3       0.40      0.45      0.42        93
           4       0.74      0.97      0.84      9711

    accuracy                           0.83     22544
   macro avg       0.78      0.66      0.68     22544
weighted avg       0.85      0.83      0.81     22544


Confusion Matrix:

[[6256  138    1    2 1063]
 [ 166 2134    6   14  747]
 [   0   16  729   48 1420]
 [   0    6   31   42   14]
 [  67  198    3    0 9443]]


In [26]:
y_pred = xgb_model_robust.predict(
    X_test_fused
)
robust_results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average='weighted'),
    "Recall": recall_score(y_test, y_pred, average='weighted'),
    "F1 Score": f1_score(y_test, y_pred, average='weighted'),
    "Macro F1": f1_score(y_test, y_pred, average='macro'),
    "Classification Report": classification_report(y_test, y_pred),
    "Confusion Matrix": confusion_matrix(y_test, y_pred),
}

print("\n====================================")
print("ROBUST MODEL RESULTS")
print("====================================")

print(f"Accuracy : {robust_results['Accuracy']:.4f}")
print(f"Precision: {robust_results['Precision']:.4f}")
print(f"Recall   : {robust_results['Recall']:.4f}")
print(f"F1 Score : {robust_results['F1 Score']:.4f}")
print(f"Macro F1 : {robust_results['Macro F1']:.4f}")

print("\nClassification Report:\n")
print(robust_results["Classification Report"])

print("\nConfusion Matrix:\n")
print(robust_results["Confusion Matrix"])


ROBUST MODEL RESULTS
Accuracy : 0.8109
Precision: 0.8409
Recall   : 0.8109
F1 Score : 0.8016
Macro F1 : 0.6906

Classification Report:

              precision    recall  f1-score   support

           0       0.96      0.81      0.88      7460
           1       0.87      0.63      0.73      3067
           2       0.92      0.37      0.53      2213
           3       0.45      0.53      0.49        93
           4       0.72      0.97      0.83      9711

    accuracy                           0.81     22544
   macro avg       0.79      0.66      0.69     22544
weighted avg       0.84      0.81      0.80     22544


Confusion Matrix:

[[6032   29   44    2 1353]
 [ 166 1944    2   11  944]
 [   0   50  815   47 1301]
 [   0   13   18   49   13]
 [  67  200    4    0 9440]]


In [29]:
ensemble_results = {}

for model in [xgb_model, catboost_model, adaboost_model]:

    y_pred = model.predict(X_test)

    ensemble_results[model.__class__.__name__] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1 Score": f1_score(y_test, y_pred, average='weighted'),
        "Macro F1": f1_score(y_test, y_pred, average='macro'),
        "Classification Report": classification_report(y_test, y_pred),
        "Confusion Matrix": confusion_matrix(y_test, y_pred)
    }

    print(f"\nModel: {model.__class__.__name__}")
    print(f"Accuracy: {ensemble_results[model.__class__.__name__]['Accuracy']:.4f}")
    print(f"Precision: {ensemble_results[model.__class__.__name__]['Precision']:.4f}")
    print(f"Recall: {ensemble_results[model.__class__.__name__]['Recall']:.4f}")
    print(f"F1 Score: {ensemble_results[model.__class__.__name__]['F1 Score']:.4f}")
    print(f"Macro F1 Score: {ensemble_results[model.__class__.__name__]['Macro F1']:.4f}")

    print(classification_report(y_test, y_pred))


Model: XGBClassifier
Accuracy: 0.8149
Precision: 0.8418
Recall: 0.8149
F1 Score: 0.8036
Macro F1 Score: 0.6715
              precision    recall  f1-score   support

           0       0.96      0.85      0.90      7460
           1       0.84      0.59      0.69      3067
           2       0.94      0.35      0.51      2213
           3       0.39      0.45      0.42        93
           4       0.73      0.97      0.83      9711

    accuracy                           0.81     22544
   macro avg       0.77      0.64      0.67     22544
weighted avg       0.84      0.81      0.80     22544


Model: CatBoostClassifier
Accuracy: 0.8044
Precision: 0.8265
Recall: 0.8044
F1 Score: 0.7935
Macro F1 Score: 0.6690
              precision    recall  f1-score   support

           0       0.96      0.81      0.88      7460
           1       0.80      0.59      0.68      3067
           2       0.86      0.36      0.51      2213
           3       0.45      0.44      0.45        93
           

## Linear Models

In [30]:
stand_scaler=load('standard_scaler.pkl')

In [31]:
if 'level' in X_test.columns:
    X_test = X_test.drop(columns=['level'])

In [32]:
print("Scaler expects:", len(stand_scaler.feature_names_in_))
print("X_test has:", len(X_test.columns))
print("Extra:", set(X_test.columns) - set(stand_scaler.feature_names_in_))

Scaler expects: 38
X_test has: 119
Extra: {'service_login', 'service_nnsp', 'service_http_443', 'service_ldap', 'service_supdup', 'service_urp_i', 'service_iso_tsap', 'service_Z39_50', 'service_hostnames', 'service_pop_2', 'service_rje', 'service_mtp', 'service_gopher', 'service_vmnet', 'service_private', 'service_http', 'protocol_type_udp', 'service_daytime', 'service_shell', 'service_discard', 'flag_S2', 'service_ftp', 'service_nntp', 'protocol_type_tcp', 'flag_REJ', 'service_http_8001', 'service_kshell', 'service_telnet', 'service_ntp_u', 'flag_RSTO', 'service_imap4', 'flag_S0', 'service_sunrpc', 'service_harvest', 'service_remote_job', 'service_tim_i', 'service_uucp', 'service_pop_3', 'service_tftp_u', 'service_domain', 'flag_RSTR', 'service_whois', 'service_netbios_ssn', 'service_ssh', 'service_finger', 'service_efs', 'service_printer', 'service_ecr_i', 'service_time', 'service_link', 'service_other', 'service_ftp_data', 'service_name', 'service_sql_net', 'flag_SH', 'service_eco_i

In [33]:
numerical_cols = X_test.select_dtypes(include=np.number).columns
X_test_scaled_linear = X_test.copy()

X_test_scaled_linear[stand_scaler.feature_names_in_] = (
    stand_scaler.transform(
        X_test_scaled_linear[stand_scaler.feature_names_in_]
    )
)

In [34]:
print(X_test_scaled_linear.shape)
print(knn_model.n_features_in_)

(22544, 119)
119


In [35]:
for model in [knn_model, logistic_model, svm_model, nb_model]:

    y_pred = model.predict(X_test_scaled_linear)
    linear_results = {}
    linear_results[model.__class__.__name__] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1 Score": f1_score(y_test, y_pred, average='weighted'),
        "Macro F1": f1_score(y_test, y_pred, average='macro'),
        "Classification Report": classification_report(y_test, y_pred),
        "Confusion Matrix": confusion_matrix(y_test, y_pred)
    }

    print(f"\nModel: {model.__class__.__name__}")
    print(f"Accuracy: {linear_results[model.__class__.__name__]['Accuracy']:.4f}")
    print(f"Precision: {linear_results[model.__class__.__name__]['Precision']:.4f}")
    print(f"Recall: {linear_results[model.__class__.__name__]['Recall']:.4f}")
    print(f"F1 Score: {linear_results[model.__class__.__name__]['F1 Score']:.4f}")
    print(f"Macro F1 Score: {linear_results[model.__class__.__name__]['Macro F1']:.4f}")
    print(classification_report(y_test, y_pred))


Model: KNeighborsClassifier
Accuracy: 0.7689
Precision: 0.8120
Recall: 0.7689
F1 Score: 0.7355
Macro F1 Score: 0.5663
              precision    recall  f1-score   support

           0       0.96      0.82      0.88      7460
           1       0.84      0.54      0.66      3067
           2       0.89      0.05      0.10      2213
           3       0.61      0.29      0.39        93
           4       0.67      0.97      0.79      9711

    accuracy                           0.77     22544
   macro avg       0.80      0.53      0.57     22544
weighted avg       0.81      0.77      0.74     22544


Model: LogisticRegression
Accuracy: 0.7633
Precision: 0.7940
Recall: 0.7633
F1 Score: 0.7406
Macro F1 Score: 0.5663
              precision    recall  f1-score   support

           0       0.92      0.81      0.86      7460
           1       0.87      0.58      0.70      3067
           2       0.82      0.14      0.24      2213
           3       0.30      0.23      0.26        93
    